# ED Simulation + Ops-Aware Thresholding + Policy Replay (Standalone)
Self-contained CPU notebook: realistic arrivals, triage mix, GRU training, calibration, PR/cost + ops-aware thresholds, policy A/B/C replays, KPIs, and audit anchor.

In [1]:

# --- Config ---
import os, json, math, random, numpy as np
from pathlib import Path

ARTS = Path("artifacts"); ARTS.mkdir(exist_ok=True, parents=True)

SEED = 1001
np.random.seed(SEED); random.seed(SEED)

ACTIONS = [
    "NO_OP","ORDER_ECG","ORDER_LABS","ORDER_XR","ORDER_CT",
    "PERFORM_FAST","REQUEST_CONSULT","REQUEST_BED"
]
COMMON_ACTIONS = ["ORDER_ECG","ORDER_LABS","ORDER_XR","ORDER_CT","PERFORM_FAST"]
PAGING_ACTIONS = ["REQUEST_CONSULT","REQUEST_BED"]

LAMBDA_BY_HOUR = {
    0:1.0,1:1.0,2:1.0,3:1.0,4:1.0,5:1.2,
    6:1.5,7:2.0,8:3.5,9:3.8,10:3.5,11:3.2,
    12:3.0,13:3.0,14:3.0,15:3.2,16:3.5,17:3.8,
    18:3.0,19:2.5,20:2.0,21:1.8,22:1.5,23:1.2
}
SIM_MINUTES = 12*60

TRIAGE_DIST = {"red":0.03, "orange":0.30, "yellow":0.55, "green":0.12}
MAX_CONCURRENT_RED = 5

LOS_MED_H = {"green":2.5, "yellow":5.0, "orange":7.0, "red":2.0}
LOS_SIGMA = 0.35

CAPACITY = {"CT_per_h_day":5, "CT_per_h_night":3, "XR_per_h":8, "FAST_per_h":999}
DAY_START, DAY_END = 8, 20

RECALL_FLOOR_COMMON = 0.80
FP_BUDGET_PER_H = {"REQUEST_CONSULT":1.0, "REQUEST_BED":0.5}

VAL_MIN = {"REQUEST_CONSULT":15, "REQUEST_BED":15}

FEATURE_NAMES = [
    "minute_of_day","triage_red","triage_orange","triage_yellow",
    "age_ge_65","on_anticoag","hi_energy_mech","gcs_lt_15","neuro_deficit",
    "o2_need_level","map_low","arrhythmia_flag","sepsis_flag"
]
N_FEATURES = len(FEATURE_NAMES)
SEQ_LEN = 5


In [2]:

# --- Arrival & triage utilities ---
def nonhom_poisson_arrivals(total_minutes, lam_by_hour):
    arrivals = []
    for m in range(total_minutes):
        h = (m//60)%24
        lam = lam_by_hour.get(h,1.0)/60.0
        if np.random.rand() < lam:
            arrivals.append(m)
    return arrivals

def sample_triage():
    r = np.random.rand()
    cum=0.0
    for k,p in TRIAGE_DIST.items():
        cum+=p
        if r<=cum:
            return k
    return "yellow"

def lognormal_from_median(med, sigma=LOS_SIGMA):
    mu = math.log(max(med, 0.1))
    return np.random.lognormal(mean=mu, sigma=sigma)


In [3]:

# --- Synthetic patient/event generator ---
def gen_patients_and_events(total_minutes=SIM_MINUTES):
    arrivals = nonhom_poisson_arrivals(total_minutes, LAMBDA_BY_HOUR)
    patients = []
    active_reds = 0
    for i, at in enumerate(arrivals):
        tr = sample_triage()
        if tr=="red" and active_reds>=MAX_CONCURRENT_RED:
            tr="orange"
        if tr=="red": active_reds += 1
        age_ge_65   = int(np.random.rand()<0.45)
        on_anticoag = int(age_ge_65 and (np.random.rand()<0.25))
        hi_energy   = int(np.random.rand()<0.15)
        gcs_low     = int(tr=="red" and np.random.rand()<0.5) or int(hi_energy and np.random.rand()<0.1)
        neuro_def   = int((hi_energy or on_anticoag) and np.random.rand()<0.1)
        o2_level    = int(np.random.choice([0,1,2], p=[0.55,0.30,0.15]))
        map_low     = int(np.random.rand()<0.12)
        arr_flag    = int(np.random.rand()<0.20)
        sepsis_flag = int(np.random.rand()<0.15)
        los_h = lognormal_from_median(LOS_MED_H[tr])
        dep_min = int(at + min(total_minutes-at, los_h*60))
        minute_of_day = at % 1440
        feats = [
            minute_of_day/1440.0,
            int(tr=="red"), int(tr=="orange"), int(tr=="yellow"),
            age_ge_65, on_anticoag, hi_energy, gcs_low, neuro_def,
            o2_level, map_low, arr_flag, sepsis_flag
        ]
        label = "NO_OP"
        if (hi_energy or gcs_low or neuro_def or on_anticoag) and (tr in ["red","orange"]):
            label = np.random.choice(["ORDER_CT","PERFORM_FAST"], p=[0.7,0.3])
        elif arr_flag or map_low:
            label = np.random.choice(["ORDER_ECG","ORDER_LABS"], p=[0.7,0.3])
        elif sepsis_flag:
            label = "ORDER_LABS"
        elif tr in ["orange","yellow"]:
            label = np.random.choice(["ORDER_XR","ORDER_LABS","ORDER_ECG"], p=[0.2,0.5,0.3])
        if label in ["ORDER_CT","PERFORM_FAST"] and (tr!="green") and (np.random.rand()<0.25):
            label = np.random.choice(["REQUEST_CONSULT","REQUEST_BED"], p=[0.7,0.3])
        patients.append({
            "id": i, "arrive_min": at, "depart_min": dep_min, "triage": tr,
            "features": feats, "label": label
        })
    X = []; y = []; ts = []
    for p in patients:
        base = np.array(p["features"], dtype=float)
        seq = np.stack([base + 0.01*np.random.randn(len(base)) for _ in range(SEQ_LEN)], axis=0)
        X.append(seq)
        y.append(ACTIONS.index(p["label"]))
        ts.append(p["arrive_min"])
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    ts = np.array(ts, dtype=np.int64)
    return X, y, ts, patients

X, y, ts, patients = gen_patients_and_events()
len(X), len(y)


(15, 15)

In [4]:

# --- Train/Val split with minima for rare classes ---
from collections import defaultdict, Counter
def train_val_split_with_min(X, y, ts, val_frac=0.17, min_per_class=VAL_MIN):
    N = len(y)
    idx = np.arange(N); np.random.shuffle(idx)
    n_val = max(1, int(N*val_frac))
    val_idx = set(idx[:n_val].tolist())
    train_idx = set(idx[n_val:].tolist())
    name_to_id = {name:i for i,name in enumerate(ACTIONS)}
    for name, req in min_per_class.items():
        cid = name_to_id[name]
        cur = [i for i in val_idx if y[i]==cid]
        need = req - len(cur)
        if need>0:
            addable = [i for i in train_idx if y[i]==cid]
            take = addable[:need]
            for t in take:
                train_idx.remove(t); val_idx.add(t)
    train_idx = np.array(sorted(list(train_idx)), dtype=int)
    val_idx   = np.array(sorted(list(val_idx)), dtype=int)
    return (X[train_idx], y[train_idx], ts[train_idx]), (X[val_idx], y[val_idx], ts[val_idx])

(trainX, trainy, traints), (valX, valy, valts) = train_val_split_with_min(X, y, ts)
print("Train/Val shapes:", trainX.shape, valX.shape)
print("Train classes:", Counter(trainy))
print("Val classes:", Counter(valy))


Train/Val shapes: (12, 5, 13) (3, 5, 13)
Train classes: Counter({2: 4, 1: 3, 0: 2, 3: 2, 5: 1})
Val classes: Counter({1: 1, 7: 1, 2: 1})


In [5]:

# --- GRU model (PyTorch) ---
import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cpu")

class GRUHead(nn.Module):
    def __init__(self, in_f, hidden=128, n_actions=8, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(in_f, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_actions)
    def forward(self, x):
        out, _ = self.gru(x)
        h = out[:,-1,:]
        h = self.drop(h)
        return self.fc(h)

def class_weights(y, n_classes):
    cnt = np.bincount(y, minlength=n_classes).astype(float)
    inv = 1.0 / np.sqrt(cnt + 1e-6)
    w = inv / inv.sum() * n_classes
    return torch.tensor(w, dtype=torch.float32)

n_actions = len(ACTIONS)
model = GRUHead(len(FEATURE_NAMES), hidden=128, n_actions=n_actions, dropout=0.2).to(device)
weights = class_weights(trainy, n_actions).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(model.parameters(), lr=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=35)

def to_t(x): return torch.tensor(x, dtype=torch.float32).to(device)
def to_y(y): return torch.tensor(y, dtype=torch.long).to(device)

BX=64
def batches(X,y,bs=BX):
    N=len(y); idx=np.arange(N); np.random.shuffle(idx)
    for i in range(0,N,bs):
        b=idx[i:i+bs]
        yield X[b], y[b]

EPOCHS = 35
for ep in range(1,EPOCHS+1):
    model.train(); tl=0.0; tn=0
    for xb,yb in batches(trainX,trainy,BX):
        xb_t, yb_t = to_t(xb), to_y(yb)
        opt.zero_grad()
        logits = model(xb_t)
        loss = criterion(logits, yb_t)
        loss.backward(); opt.step()
        tl += float(loss.detach())*len(yb); tn += len(yb)
    model.eval()
    with torch.no_grad():
        val_logits = model(to_t(valX))
        vl = float(criterion(val_logits, to_y(valy)).detach())
    sched.step()
    if ep%5==0 or ep in [1,2,3]:
        print(f"Epoch {ep:02d}: train={tl/tn:.4f} val={vl:.4f}")


Epoch 01: train=2.1277 val=2.1037
Epoch 02: train=2.1092 val=2.1126
Epoch 03: train=2.1055 val=2.1219
Epoch 05: train=2.0830 val=2.1408
Epoch 10: train=2.0245 val=2.1850
Epoch 15: train=1.9851 val=2.2254
Epoch 20: train=1.9330 val=2.2573
Epoch 25: train=1.9313 val=2.2780
Epoch 30: train=1.8963 val=2.2871
Epoch 35: train=1.8985 val=2.2887


In [6]:

# --- Calibration (temperature scaling) & val cache ---
with torch.no_grad():
    val_logits = model(to_t(valX)).cpu().numpy()

def softmax(z, T=1.0, axis=1):
    zT = z / max(T, 1e-6)
    zT -= zT.max(axis=axis, keepdims=True)
    e = np.exp(zT)
    return e / e.sum(axis=axis, keepdims=True)

def nll(probs, y_true):
    idx = np.arange(len(y_true))
    p = probs[idx, y_true]
    return -np.log(np.clip(p,1e-9,1.0)).mean()

Ts = np.linspace(0.6,1.8,25)
bestT, bestnll = 1.0, 1e9
for T in Ts:
    probs = softmax(val_logits, T=T)
    cur = nll(probs, valy)
    if cur<bestnll:
        bestnll, bestT = cur, T
print(f"[Calib] Best temperature T={bestT:.3f} (grid)")
val_probs = softmax(val_logits, T=bestT)

ARTS.mkdir(exist_ok=True, parents=True)
np.savez(ARTS/"val_cache.npz", probs=val_probs.astype("float32"), y_true=valy.astype("int64"),
         action_names=np.array(ACTIONS, dtype=object), ts=valts.astype("int64"), T=bestT)
print("Saved", ARTS/"val_cache.npz")


[Calib] Best temperature T=0.600 (grid)
Saved artifacts/val_cache.npz


In [7]:

# --- Threshold selection (PR-cost + ops-aware) ---
def pick_tau_cost(y_true_bin, y_prob, fp_cost=1.0, fn_cost=5.0):
    taus = np.linspace(0,1,101)
    best=None
    for t in taus:
        yp = (y_prob>=t).astype(int)
        tp = int(((yp==1)&(y_true_bin==1)).sum())
        fp = int(((yp==1)&(y_true_bin==0)).sum())
        fn = int(((yp==0)&(y_true_bin==1)).sum())
        prec = tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9)
        f1 = 2*prec*rec/(prec+rec+1e-9)
        cost = fp_cost*fp + fn_cost*fn
        if (best is None) or (cost<best[0]):
            best=(cost,t,prec,rec,f1,tp,fp,fn)
    c,t,prec,rec,f1,tp,fp,fn = best
    return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
            "cost":float(c),"tp":tp,"fp":fp,"fn":fn}

def pick_tau_ops(y_true_bin, y_prob, ts_min, recall_floor=None, fp_budget_per_h=None):
    taus = np.linspace(0,1,101)
    hours = (ts_min.max()-ts_min.min()+1)/60.0
    best=None
    for t in taus:
        yp = (y_prob>=t).astype(int)
        tp = int(((yp==1)&(y_true_bin==1)).sum())
        fp = int(((yp==1)&(y_true_bin==0)).sum())
        fn = int(((yp==0)&(y_true_bin==1)).sum())
        prec = tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9)
        f1 = 2*prec*rec/(prec+rec+1e-9)
        fp_per_h = fp/max(hours,1e-6)
        if recall_floor is not None and rec < recall_floor:
            continue
        if fp_budget_per_h is not None and fp_per_h > fp_budget_per_h:
            continue
        score = f1
        if (best is None) or (score>best[0]):
            best=(score,t,prec,rec,f1,tp,fp,fn,fp_per_h)
    if best is None:
        t=0.5; yp=(y_prob>=t).astype(int)
        tp = int(((yp==1)&(y_true_bin==1)).sum())
        fp = int(((yp==1)&(y_true_bin==0)).sum())
        fn = int(((yp==0)&(y_true_bin==1)).sum())
        prec = tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9)
        f1 = 2*prec*rec/(prec+rec+1e-9); fp_per_h = fp/((ts_min.max()-ts_min.min()+1)/60.0)
        return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
                "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h),"status":"fallback"}
    score,t,prec,rec,f1,tp,fp,fn,fp_per_h = best
    return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
            "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h)}

dat = np.load(ARTS/"val_cache.npz", allow_pickle=True)
probs, y_true, action_names, ts_min = dat["probs"], dat["y_true"], dat["action_names"].tolist(), dat["ts"]

thr_cost = {}
for ci,name in enumerate(action_names):
    yb = (y_true==ci).astype(int)
    thr_cost[name] = pick_tau_cost(yb, probs[:,ci])
json.dump(thr_cost, open(ARTS/"thresholds.json","w"), indent=2)
print("Wrote thresholds.json")

thr_ops, notes = {}, {}
for ci,name in enumerate(action_names):
    yb = (y_true==ci).astype(int)
    recall_floor = RECALL_FLOOR_COMMON if name in COMMON_ACTIONS else None
    fp_budget = FP_BUDGET_PER_H.get(name, None) if name in PAGING_ACTIONS else None
    info = pick_tau_ops(yb, probs[:,ci], ts_min, recall_floor, fp_budget)
    thr_ops[name] = info
    notes[name] = {"recall_floor": recall_floor, "fp_budget_per_h": fp_budget}
json.dump(thr_ops, open(ARTS/"thresholds_ops.json","w"), indent=2)
json.dump(notes, open(ARTS/"thresholds_ops_notes.json","w"), indent=2)
print("Wrote ops thresholds")


Wrote thresholds.json
Wrote ops thresholds


In [8]:

# --- Policy + A/B/C Replays ---
from collections import defaultdict

def capacity_fresh(ts_now, last_update_ts, max_age_min=20):
    if last_update_ts is None: return False
    return (ts_now - last_update_ts) <= max_age_min

def simulate_capacity_updates(total_minutes=SIM_MINUTES):
    updates = []
    t = 0
    while t<total_minutes:
        updates.append(t)
        t += int(np.random.uniform(10,20))
    return updates

def policy_blocks(state, action_name, pkt):
    if action_name in ["REQUEST_CONSULT","REQUEST_BED"]:
        if not pkt.get("packet_complete", False):
            return False, "packet_incomplete"
        if not pkt.get("trauma_cleared", True):
            return False, "ownership"
        if not capacity_fresh(pkt["ts_now"], state.get("last_capacity_update_ts", None)):
            return False, "capacity_stale"
        key = (pkt["patient_id"], action_name, pkt.get("target_service","IM"))
        if key in state["sent_packets"]:
            return False, "duplicate"
    return True, None

def make_packet(patient_id, action_name, ts_now, clinical_flags):
    required = ["route_reason","vitals_ok","labs_ok"]
    complete = all(clinical_flags.get(k, False) for k in required)
    pkt = {"patient_id":patient_id,"ts_now":ts_now,"target_service":"IM","packet_complete":complete,
           "trauma_cleared": clinical_flags.get("trauma_cleared", True)}
    return pkt

import torch
def replay(mode="A", thresholds=None):
    assert thresholds is not None
    logs = {"proposals":[], "gate_blocks":[], "policy_blocks":[], "approvals":[]}
    state = {"last_capacity_update_ts":None, "sent_packets":set()}
    cap_updates = set(simulate_capacity_updates())
    order = np.argsort(ts)
    T = float(np.load(ARTS/'val_cache.npz', allow_pickle=True)['T'])
    for idx in order:
        tnow = int(ts[idx])
        if tnow in cap_updates:
            state["last_capacity_update_ts"] = tnow
        with torch.no_grad():
            logits = model(to_t(X[idx:idx+1]))
            pr = torch.softmax(logits/T, dim=-1).cpu().numpy()[0]
        picks = []
        for ci,name in enumerate(ACTIONS):
            tau = thresholds.get(name, {}).get("tau", 0.5)
            if pr[ci] >= tau:
                picks.append((name, float(pr[ci])))
        if not picks: 
            continue
        picks_sorted = sorted(picks, key=lambda x: (x[0]=="NO_OP", -x[1]))
        name, score = picks_sorted[0]
        tau = thresholds.get(name, {}).get("tau", 0.5)
        if score < tau:
            logs["gate_blocks"].append({"action":name,"p":score,"tau":tau,"reason":"below_tau"}); continue
        if mode in ["B","C"] and name in ["REQUEST_CONSULT","REQUEST_BED"]:
            pkt = make_packet(patient_id=int(idx), action_name=name, ts_now=tnow,
                              clinical_flags={"route_reason":True,"vitals_ok":True,"labs_ok":True,"trauma_cleared":True})
            ok, reason = policy_blocks(state, name, pkt)
            if not ok:
                logs["policy_blocks"].append({"action":name,"reason":reason}); continue
            state["sent_packets"].add((pkt["patient_id"], name, pkt["target_service"]))
        logs["proposals"].append({"idx":int(idx),"action":name,"p":score,"ts":tnow})
        if name in PAGING_ACTIONS:
            if np.random.rand()<0.7:
                logs["approvals"].append({"idx":int(idx),"action":name,"ts":tnow})
        else:
            logs["approvals"].append({"idx":int(idx),"action":name,"ts":tnow})
    return logs

thr_cost = json.load(open(ARTS/"thresholds.json"))
thr_ops  = json.load(open(ARTS/"thresholds_ops.json"))

logsA = replay("A", thresholds=thr_cost)  # policy OFF, PR-cost
logsB = replay("B", thresholds=thr_cost)  # policy ON, PR-cost
logsC = replay("C", thresholds=thr_ops)   # policy ON, ops-aware

def summarize(logs, label):
    props = len(logs["proposals"]); appr = len(logs["approvals"])
    print(f"{label}: proposals={props} approvals={appr} approve_rate={appr/max(1,props):.2f} "
          f"gate_blocks={len(logs['gate_blocks'])} policy_blocks={len(logs['policy_blocks'])}")
summarize(logsA,"Replay A (policy OFF, PR-cost)")
summarize(logsB,"Replay B (policy ON, PR-cost)")
summarize(logsC,"Replay C (policy ON, ops)")

json.dump(logsA, open(ARTS/"replay_A.json","w"), indent=2)
json.dump(logsB, open(ARTS/"replay_B.json","w"), indent=2)
json.dump(logsC, open(ARTS/"replay_C.json","w"), indent=2)
print("Saved replay logs under artifacts/")


Replay A (policy OFF, PR-cost): proposals=15 approvals=14 approve_rate=0.93 gate_blocks=0 policy_blocks=0
Replay B (policy ON, PR-cost): proposals=11 approvals=11 approve_rate=1.00 gate_blocks=0 policy_blocks=4
Replay C (policy ON, ops): proposals=13 approvals=13 approve_rate=1.00 gate_blocks=0 policy_blocks=2
Saved replay logs under artifacts/


In [9]:

# --- KPIs ---
def kpis_from_logs(logs):
    props = logs["proposals"]; appr = logs["approvals"]
    approve_rate = len(appr)/max(1,len(props))
    saved = 0.0
    for p in props:
        if p["action"] in ["ORDER_ECG","ORDER_LABS"]: saved += 1.5
        elif p["action"] in ["ORDER_CT","PERFORM_FAST"]: saved += 4.0
        elif p["action"] in ["REQUEST_CONSULT","REQUEST_BED"]: saved += 3.0
    median_minutes_saved = saved / max(1,len(props))
    page_props = [p for p in props if p["action"] in PAGING_ACTIONS]
    page_appr  = [a for a in appr  if a["action"] in PAGING_ACTIONS]
    fpr = (len(page_props)-len(page_appr))/max(1,len(page_props))
    return {"proposals":len(props),"approvals":len(appr),"approve_rate":round(approve_rate,3),
            "false_page_rate":round(fpr,3),"median_minutes_saved":round(median_minutes_saved,2)}

kpiA = kpis_from_logs(logsA)
kpiB = kpis_from_logs(logsB)
kpiC = kpis_from_logs(logsC)

json.dump({"A":kpiA,"B":kpiB,"C":kpiC}, open(ARTS/"kpis.json","w"), indent=2)
print("KPIs:", {"A":kpiA,"B":kpiB,"C":kpiC})


KPIs: {'A': {'proposals': 15, 'approvals': 14, 'approve_rate': 0.933, 'false_page_rate': 0.25, 'median_minutes_saved': 2.07}, 'B': {'proposals': 11, 'approvals': 11, 'approve_rate': 1.0, 'false_page_rate': 0.0, 'median_minutes_saved': 1.73}, 'C': {'proposals': 13, 'approvals': 13, 'approve_rate': 1.0, 'false_page_rate': 0.0, 'median_minutes_saved': 1.38}}


In [10]:

# --- Audit anchor ---
import hashlib, time, glob
def file_hash(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

asset_paths = sorted([str(p) for p in Path("artifacts").glob("*.*")])
leaves = [file_hash(p) for p in asset_paths]
layer = leaves[:]
if not layer: root = hashlib.sha256(b"").hexdigest()
else:
    while len(layer)>1:
        nxt = []
        it = iter(layer)
        for a in it:
            b = next(it, a)
            nxt.append(hashlib.sha256((a+b).encode()).hexdigest())
        layer = nxt
    root = layer[0]

anchor = {"root":root,"ts":time.strftime("%Y-%m-%dT%H:%M:%SZ")}
with open("artifacts/audit_anchor.log","a") as f:
    f.write(json.dumps(anchor)+"\n")
print("Anchored:", anchor)


Anchored: {'root': 'baeb577da0da7ee76a2cfa2e9f1b58640264b401fd355bfdfedc103e856ee2ce', 'ts': '2025-08-14T03:22:14Z'}
